# Vẽ biểu đồ kết quả thực nghiệm
So sánh DQN / Double DQN / Dijkstra baseline từ các CSV trong . Hỗ trợ nhiều seed (mean ± std).

In [ ]:
%matplotlib inline
import glob, os, re
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
CSV_DIR="results/csv"
FIG_DIR="results/figures"
os.makedirs(FIG_DIR, exist_ok=True)

In [ ]:
def parse_tag(path):
    name = os.path.basename(path).replace(".csv", "")
    m = re.match(r"(dqn|ddqn)_load([0-9.]+)(_s([0-9]+))?", name)
    return (m.group(1), float(m.group(2)), int(m.group(4) or 0)) if m else None

training = {}
for f in glob.glob(f"{CSV_DIR}/dqn*.csv") + glob.glob(f"{CSV_DIR}/ddqn*.csv"):
    tag = parse_tag(f)
    if tag is None: continue
    algo, load, seed = tag
    df = pd.read_csv(f); df["algo"]=algo; df["seed"]=seed
    training.setdefault(load, []).append(df)
baseline = pd.read_csv(f"{CSV_DIR}/baseline.csv") if os.path.exists(f"{CSV_DIR}/baseline.csv") else None
print("loads:", sorted(training.keys()))

In [ ]:
def ms(dfs, col, w=20):
    m = pd.concat([d[["episode", col]].dropna() for d in dfs], ignore_index=True)
    g = m.groupby("episode")[col].agg(["mean","std"]).reset_index()
    return g["episode"].values, g["mean"].rolling(w, min_periods=1).mean().values, g["std"].rolling(w, min_periods=1).mean().values

colors = {"dqn": "#1f77b4", "ddqn": "#ff7f0e"}
for load, dfs in training.items():
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    specs = [(axes[0,0],"train_reward","Reward/episode"), (axes[0,1],"train_loss","Loss"),
             (axes[1,0],"train_avg_delay_ms","Avg delay (ms)"), (axes[1,1],"train_loss_rate","Packet loss rate")]
    for algo in ["dqn","ddqn"]:
        sub = [d for d in dfs if d["algo"].iloc[0]==algo]
        if not sub: continue
        c = colors[algo]
        for ax, col, t in specs:
            x, y, s = ms(sub, col)
            ax.plot(x, y, label=algo, color=c); ax.fill_between(x, y-s, y+s, color=c, alpha=0.15); ax.set_title(t)
        for d in sub:
            ev = d.dropna(subset=["eval_reward"])
            if len(ev): axes[0,0].plot(ev["episode"], ev["eval_reward"], "o", color=c, alpha=0.4)
    for ax in axes.flat: ax.set_xlabel("episode"); ax.legend()
    fig.suptitle(f"Training curves (load={load}, mean±std over seeds)")
    fig.tight_layout(); fig.savefig(f"{FIG_DIR}/training_load{load}.png", dpi=150)
    plt.show()

In [ ]:
rows = []
for load, dfs in training.items():
    for algo in ["dqn","ddqn"]:
        for d in [x for x in dfs if x["algo"].iloc[0]==algo]:
            ev = d.dropna(subset=["eval_reward"])
            if len(ev)==0: continue
            last = ev.tail(1).iloc[0]
            rows.append({"load": load, "algo": algo, "seed": d["seed"].iloc[0],
                         "reward": last["eval_reward"], "delay_ms": last["eval_avg_delay_ms"],
                         "loss_rate": last["eval_loss_rate"], "throughput": last["eval_throughput"]})
if baseline is not None:
    for _, r in baseline.iterrows():
        rows.append({"load": r["load"], "algo": "dijkstra", "seed": 0, "reward": r["episode_reward"],
                     "delay_ms": r["avg_delay_ms"], "loss_rate": r["packet_loss_rate"], "throughput": r["throughput"]})
df = pd.DataFrame(rows)
g = df.groupby(["load","algo"])
summary = pd.DataFrame({"reward": g["reward"].mean(), "reward_std": g["reward"].std(),
    "delay_ms": g["delay_ms"].mean(), "delay_ms_std": g["delay_ms"].std(),
    "loss_rate": g["loss_rate"].mean(), "loss_rate_std": g["loss_rate"].std(),
    "throughput": g["throughput"].mean(), "throughput_std": g["throughput"].std(),
    "n_seeds": g.size()}).reset_index()
summary.to_csv(f"{CSV_DIR}/summary.csv", index=False)
summary

In [ ]:
metrics = [("reward","reward_std"), ("delay_ms","delay_ms_std"), ("loss_rate","loss_rate_std"), ("throughput","throughput_std")]
for load in sorted(summary["load"].unique()):
    sub = summary[summary["load"]==load]
    algos = sorted(sub["algo"].unique())
    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax, (m, ms_) in zip(axes, metrics):
        for i, algo in enumerate(algos):
            r = sub[sub["algo"]==algo].iloc[0]
            ax.bar(i, r[m], width=0.5, yerr=r.get(ms_,0) or 0, capsize=4, label=algo)
        ax.set_title(m); ax.set_xticks(range(len(algos))); ax.set_xticklabels(algos); ax.legend()
    fig.suptitle(f"Final comparison (load={load})")
    fig.tight_layout(); fig.savefig(f"{FIG_DIR}/comparison_load{load}.png", dpi=150)
    plt.show()